# Lab 6 — *Sound of Muzak*: one rhythmic object through many representations

This is the flagship case study. It uses **Bruno Di Giorgi's original synthetic WAV reconstruction** of the drum pattern, not the commercial Porcupine Tree recording. The symbolic pattern is retained so that the same rhythmic object can be examined as code, event grid, audio, and signal-derived representation.

For listening comparison, use the external reference supplied for the course: https://youtu.be/cAtsmJMw0kU?si=ylsjelUjdi_f4_Ou

The point of the lab is to follow the *same rhythmic object* from symbolic structure to audio and back to signal-derived representations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from IPython.display import Audio, display

np.set_printoptions(precision=3, suppress=True)

def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, dtype=float).reshape(-1)
    if max_lag is None:
        max_lag = len(x)-1
    max_lag = min(int(max_lag), len(x)-1)
    r = np.empty(max_lag+1)
    for tau in range(max_lag+1):
        a=x[:len(x)-tau]; b=x[tau:]
        r[tau] = np.dot(a,b)/max(1,len(a))
    if normalize and r[0] > 0:
        r = r/r[0]
    return r

def phase_autocorrelation(x, lags, unbiased=True):
    x=np.asarray(x,dtype=float).reshape(-1)
    lags=np.asarray(lags,dtype=int)
    P=np.zeros((len(lags), int(lags.max())))
    C=np.zeros_like(P)
    N=len(x)
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            idx=np.arange(phi, N-tau, tau, dtype=int)
            if len(idx):
                P[ir,phi]=np.sum(x[idx]*x[idx+tau])
                C[ir,phi]=len(idx)
    if unbiased:
        return P/np.where(C==0,1,C), C
    return P, C

def frame_signal(x, frame_length, hop_length, center=True):
    x=np.asarray(x,float).reshape(-1)
    if center:
        p=frame_length//2
        x=np.pad(x,(p,p))
    starts=np.arange(0,max(1,len(x)-frame_length+1),hop_length,dtype=int)
    return np.stack([x[s:s+frame_length] for s in starts],axis=1), starts

def rhythmogram(x, frame_length=256, hop_length=32, lag_min=8, lag_max=96):
    frames, starts = frame_signal(x, frame_length, hop_length, center=True)
    lags=np.arange(lag_min,lag_max+1)
    R=np.zeros((len(lags),frames.shape[1]))
    w=np.hanning(frame_length)
    for j in range(frames.shape[1]):
        f=(frames[:,j]-frames[:,j].mean())*w
        ac=unbiased_autocorrelation(f, lag_max, normalize=False)
        R[:,j]=np.maximum(ac[lags],0)
        m=R[:,j].max()
        if m>0: R[:,j]/=m
    return R,lags,starts

def dp_tempo_path(R, lags, smoothness=30.0):
    R=np.asarray(R,float)
    eps=1e-10
    obs=-np.log(R+eps)
    nP,nT=R.shape
    cost=np.full((nP,nT),np.inf); back=np.zeros((nP,nT),dtype=int)
    cost[:,0]=obs[:,0]
    lp=np.log(np.asarray(lags,float))
    trans=smoothness*(lp[:,None]-lp[None,:])**2
    for t in range(1,nT):
        M=cost[:,t-1][None,:]+trans
        back[:,t]=np.argmin(M,axis=1)
        cost[:,t]=obs[:,t]+M[np.arange(nP),back[:,t]]
    path=np.zeros(nT,dtype=int); path[-1]=np.argmin(cost[:,-1])
    for t in range(nT-2,-1,-1): path[t]=back[path[t+1],t+1]
    return np.asarray(lags)[path]

def simple_novelty(x, fs, n_fft=1024, hop=256):
    f,t,Z=signal.stft(x,fs=fs,window='hann',nperseg=n_fft,noverlap=n_fft-hop,boundary=None,padded=False)
    mag=np.abs(Z)
    flux=np.maximum(mag[:,1:]-mag[:,:-1],0).sum(axis=0)
    flux=np.r_[0,flux]
    if flux.max()>0: flux/=flux.max()
    return flux,t

PATTERN = {
    'Splash': 'S[16] 6---------------------------',
    'Closed hi-hat': 'S[16] --5-9-5-9-5-9-5-9-5-9-5-----',
    'Open hi-hat': 'S[16] ------------------------9---',
    'Snare': 'S[16] -----9------9------9------9-',
    'Kick': 'S[16] 9--9---9--9---9--9---9--9---',
}
def symbols(line): return line.split(']',1)[1].strip().replace(' ','')

def load_synthetic_wav():
    """Load Bruno Di Giorgi's synthetic Sound of Muzak WAV.

    Locally, the file is expected in ../assets or ./assets.
    In Colab, once the GitHub repository is public, it is downloaded
    automatically from the repository if it is not already present.
    """
    from pathlib import Path
    from scipy.io import wavfile
    from urllib.request import urlretrieve

    filename = "sound_of_muzak_synthetic.wav"
    candidates = [
        Path("../assets") / filename,
        Path("assets") / filename,
        Path(filename),
        Path("/content") / filename,
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        url = (
            "https://raw.githubusercontent.com/augusto-sarti/"
            "CMRM-Virtual-Rhythm-Lab/main/assets/" + filename
        )
        target = Path("/content") / filename
        try:
            urlretrieve(url, target)
            path = target
        except Exception as exc:
            raise FileNotFoundError(
                "The synthetic WAV was not found locally and could not be downloaded. "
                "While the repository is private, upload sound_of_muzak_synthetic.wav "
                "to the Colab session using the Files panel. Once the repository is public, "
                "this cell will download it automatically."
            ) from exc

    fs, x = wavfile.read(path)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if np.issubdtype(x.dtype, np.integer):
        info = np.iinfo(x.dtype)
        scale = max(abs(info.min), info.max)
        x = x.astype(np.float64) / scale
    else:
        x = x.astype(np.float64)
    return x, fs, path


## 1. The rhythmic object: from code to a common temporal grid

Bruno's original compact representation is useful because it stores several drum layers in a very small amount of text.  However, it should not be confused with musical notation.  Every character after `S[16]` occupies **one cell of the same 28-cell sixteenth-note grid**; `-` means no event, while a digit marks an event and encodes its relative strength.

The essential visual requirement is therefore that all five rows have **exactly the same length and the same right boundary**.  The following cell checks this automatically and prints the pattern with fixed-width labels and an explicit cell ruler.

In [ ]:
# Human-readable, automatically aligned view of Bruno's compact encoding
seq = {name: symbols(line) for name, line in PATTERN.items()}
lengths = {name: len(s) for name, s in seq.items()}
assert len(set(lengths.values())) == 1, f"Pattern rows do not have equal length: {lengths}"
L = next(iter(lengths.values()))

label_w = max(len(name) for name in seq)
print(" " * (label_w + 3) + "0000000000111111111122222222"[:L])
print(" " * (label_w + 3) + "0123456789012345678901234567"[:L])
for name, s in seq.items():
    print(f"{name:<{label_w}} | {s} |")
print(" " * (label_w + 3) + "^" + " " * (L-2) + "^")
print(" " * (label_w + 3) + "0" + " " * (L-2) + f"{L-1}")
print(f"\nAll rows contain {L} grid cells and terminate at the same right boundary.")

### Event-grid view

The same object can now be displayed without the compact character encoding.  Horizontal position is the common temporal grid; vertical position identifies the instrumental layer.  Marker size reflects the digit in Bruno's encoding, so stronger events remain visually distinguishable.

This representation makes the **shared 28-cell support** explicit and prevents the variable-length instrument names from creating a misleading impression of unequal rhythmic duration.

In [ ]:
# Graphical event grid: all layers share exactly the same 28-cell support
names = list(PATTERN)
seq = [symbols(PATTERN[name]) for name in names]
L = len(seq[0])

fig, ax = plt.subplots(figsize=(13, 4.6))
for y, (name, s) in enumerate(zip(names, seq)):
    # light cell centres and common vertical grid
    ax.hlines(y, -0.5, L-0.5, linewidth=0.7, alpha=0.35)
    for i, ch in enumerate(s):
        if ch.isdigit():
            strength = int(ch) / 9.0
            ax.scatter(i, y, s=55 + 150*strength, zorder=3)

for xg in range(L+1):
    ax.axvline(xg-0.5, linewidth=0.35, alpha=0.18)
# quarter-note boundaries on a sixteenth-note grid
for xg in range(0, L+1, 4):
    ax.axvline(xg-0.5, linewidth=1.0, alpha=0.45)

ax.set_xlim(-0.5, L-0.5)
ax.set_ylim(-0.7, len(names)-0.3)
ax.set_yticks(range(len(names)), names)
ax.set_xticks(range(L))
ax.set_xlabel('common sixteenth-note grid cell')
ax.set_title('Synthetic Sound of Muzak drum pattern — aligned 28-cell event grid')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Why keep the compact encoding?

The compact strings are still useful: they are the bridge between a symbolic description and executable synthesis.  In this course, however, they are treated as **one representation among several**.  We will move repeatedly among:

\[
\text{compact program representation}
\;\longleftrightarrow\;
\text{event grid}
\;\longleftrightarrow\;
\text{synthetic audio}
\;\longleftrightarrow\;
\text{signal-derived representations}.
\]

That change of viewpoint is itself part of the subject of *Computer Music Representation and Modelling*.

### Listen to Bruno's synthetic reconstruction

Before separating the rows analytically, listen first for the **compound gesture**. The audio below is the synthetic WAV produced in Bruno Di Giorgi's original teaching/research material; it is **not re-synthesized by this notebook**. This lets the analysis operate on the same reference signal that accompanied the original experiments.

A useful course hypothesis is that skilled performance need not be modelled perceptually or motorically as five completely independent clocks: the components can be grouped into a higher-level rhythmic cluster.


In [ ]:
x, fs, wav_path = load_synthetic_wav()
print(f'Loaded: {wav_path}')
print(f'Sample rate: {fs} Hz | duration: {len(x)/fs:.2f} s | peak: {np.max(np.abs(x)):.3f}')
display(Audio(x, rate=fs))
plt.figure(figsize=(12,2.5))
plt.plot(np.arange(len(x))/fs, x, lw=.5)
plt.xlim(0, min(10, len(x)/fs))
plt.xlabel('time [s]')
plt.title("Bruno Di Giorgi's synthetic Sound of Muzak reconstruction")
plt.show()


## 2. Audio → onset evidence

In [ ]:
odf,todf=simple_novelty(x,fs,n_fft=1024,hop=256)
plt.figure(figsize=(12,3)); plt.plot(todf,odf); plt.xlabel('time [s]'); plt.ylabel('novelty'); plt.title('Spectral-flux onset evidence'); plt.show()

## 3. Global autocorrelation and APM

In [ ]:
maxlag=min(220,len(odf)-1)
r=unbiased_autocorrelation(odf,maxlag)
plt.figure(figsize=(11,3)); plt.plot(r); plt.xlabel('lag [ODF samples]'); plt.title('Global autocorrelation'); plt.show()
lags=np.arange(8,min(90,maxlag+1))
P,_=phase_autocorrelation(odf,lags,unbiased=True)
plt.figure(figsize=(10,6)); plt.imshow(P,origin='lower',aspect='auto',extent=[0,P.shape[1]-1,lags[0],lags[-1]]); plt.xlabel('phase φ'); plt.ylabel('lag τ'); plt.title('Phase autocorrelation'); plt.colorbar(); plt.show()

## 4. Time-varying periodicity and a tempo path

In [ ]:
R,lagsR,starts=rhythmogram(odf,frame_length=min(256,max(64,len(odf)//2)),hop_length=12,lag_min=8,lag_max=min(80,max(12,len(odf)//3)))
path=dp_tempo_path(R,lagsR,smoothness=25)
plt.figure(figsize=(12,5)); plt.imshow(R,origin='lower',aspect='auto',extent=[0,R.shape[1]-1,lagsR[0],lagsR[-1]]); plt.plot(np.arange(len(path)),path,'w-',lw=2,label='DP period path'); plt.xlabel('frame'); plt.ylabel('period'); plt.legend(); plt.title('Rhythmogram and tracked period'); plt.show()

## 5. Return to the algebra

Earlier in the chapter, *Sound of Muzak* is used to discuss a systemic \(7{:}4\) organization and a translated component (coset) on a common temporal lattice. The signal-analysis representations do not recover that algebraic description by magic; rather, they expose periodic and phase regularities from which structural hypotheses can be formed.

**Question:** which aspects of the symbolic cluster are visible in autocorrelation, which require phase, and which depend on higher-level musical interpretation?

### Attribution

The symbolic drum pattern, the **synthetic WAV used for analysis**, and the original phase/rhythm-analysis experiments derive from material developed by **Bruno Di Giorgi** during his PhD at Politecnico di Milano. Bruno has granted permission for this material to be reused and modernized for the CMRM project. This notebook is a modern CMRM teaching reconstruction. The commercial Porcupine Tree recording is not redistributed; the external YouTube link is provided only as a listening reference.
